# Framing Annotation — Step 2: Improve Coverage of Draft Annotation Instructions

Step 2.1 tests the annotation instructions on a small random sample and lets us inspect results.
Step 2.2 runs multiple annotation passes at higher temperature to surface hard cases where the model disagrees with itself.

The goal at this point is not high reliability. We just want to make sure the instructions cover the range of cases in the data. Hard cases and mislabeled examples tell us where to add decision rules.

## Setup

In [1]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO — check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


## Annotation Instructions

We use three labels instead of two. The binary CRIME_FRAME / NO_CRIME_FRAME scheme from the first draft collapsed the direction of the frame. Since the literature distinguishes between frames that present groups as threats versus frames that present security measures as beneficial, we separate these into SECURITY_NEGATIVE and SECURITY_POSITIVE.

Adjust the `instructions` and `reminder` variables below after each inspection round.

In [2]:
instructions = """Sie sind ein neutraler Annotationsassistent. Ihre Aufgabe ist es, einen einzelnen Zeitungsabsatz zu klassifizieren. Beurteilen Sie ausschliesslich den vorliegenden Text.

Die Daten stammen aus deutschen Zeitungsartikeln. Verfuegbare Felder sind article_id, pub, par_index, text_block und group. Verwenden Sie text_block als Grundlage der Annotation. Gruppenidentitaeten sind anonymisiert und erscheinen als [Gruppe 1] oder [andere Gruppe].

Bestimmen Sie zunaechst, ob der Absatz einen expliziten Sicherheits- oder Kriminalitaetsrahmen enthaelt.

NOT_SECURITY_CRIME: Der Text rahmt das Thema, die Gruppe, die Massnahme oder das Ereignis nicht explizit in Begriffen von Kriminalitaet, oeffentlicher Sicherheit, Bedrohung, Gefahr, Polizeiarbeit, Terrorismus, Extremismus, Unordnung, Grenzkontrolle oder Schutz.

Was nicht zaehlt:
- Allgemeine Kritik oder Negativitaet ohne Sicherheits- oder Kriminalitaetsbezug
- Neutrale Erwaehnung von Polizei, Gerichten oder Gesetzen ohne zentrales Bedrohungs- oder Kriminalitaetsmotiv
- Moralische Missbilligung ohne Sicherheitskomponente
- Verwaltungsrechtliche oder prozedurale Diskussion von Gesetzen

SECURITY_NEGATIVE: Der Text rahmt die Gruppe, Massnahme oder das Ereignis als Gefahr, Bedrohung, Last, Kriminalitaetsquelle, Instabilitaetsfaktor, Gewaltrisiko oder oeffentliches Sicherheitsproblem.

SECURITY_POSITIVE: Der Text rahmt die Gruppe, Massnahme oder das Ereignis als Verbesserung der Sicherheit, Schutz, Kriminalitaetsreduzierung, Wiederherstellung der Ordnung oder erfolgreiche Bewaeltigung von Sicherheitsbedrohungen.

Entscheidungsregeln:
- Wenn [Gruppe 1] oder [andere Gruppe] Opfer einer Straftat ist: NOT_SECURITY_CRIME
- Wenn Kriminalitaet auf Armut oder soziale Ursachen zurueckgefuehrt wird, nicht auf die Gruppe selbst: NOT_SECURITY_CRIME
- Wenn der Absatz einen Sicherheitsrahmen explizit in Frage stellt oder widerlegt: NOT_SECURITY_CRIME
- Neutrale Polizeistatistiken ohne Gruppenzuschreibung: NOT_SECURITY_CRIME"""

reminder = "Klassifizieren Sie den Absatz als NOT_SECURITY_CRIME, SECURITY_NEGATIVE oder SECURITY_POSITIVE. Vergeben Sie SECURITY_NEGATIVE nur, wenn die Gruppe oder Massnahme explizit als Bedrohung oder Kriminalitaetsquelle dargestellt wird. Vergeben Sie SECURITY_POSITIVE nur, wenn Sicherheit oder Schutz als positive Leistung gerahmt wird."

print("Instructions loaded.")

Instructions loaded.


## Core Functions

`annotate()` sends a single paragraph to the API. `parse_label()` handles misspellings and label drift from small LLMs — it checks NOT_SECURITY_CRIME first because SECURITY is a substring of both other labels. If more than 5% of outputs come back as UNCLEAR, adjust the reminder.

In [3]:
VALID_LABELS = ["NOT_SECURITY_CRIME", "SECURITY_NEGATIVE", "SECURITY_POSITIVE"]


def annotate(text, instructions, reminder, temperature=0.0):
    prompt = (
        f"{instructions}\n\n"
        f"Annotieren Sie jetzt diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NOT_SECURITY_CRIME oder SECURITY_NEGATIVE oder SECURITY_POSITIVE>\n"
        "Explanation: <ein Satz, der Ihre Entscheidung begruendet>"
    )

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {
                "role": "system",
                "content": "Sie sind ein Experte fuer Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen und antworten Sie immer im angegebenen Format."
            },
            {"role": "user", "content": prompt}
        ]
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw_output):
    raw_upper = raw_output.upper()
    # order matters: check NOT_ first, then the two positives
    if "NOT_SECURITY_CRIME" in raw_upper or "NOT SECURITY CRIME" in raw_upper:
        return "NOT_SECURITY_CRIME"
    elif "SECURITY_NEGATIVE" in raw_upper or "SECURITY NEGATIVE" in raw_upper:
        return "SECURITY_NEGATIVE"
    elif "SECURITY_POSITIVE" in raw_upper or "SECURITY POSITIVE" in raw_upper:
        return "SECURITY_POSITIVE"
    else:
        return "UNCLEAR"


def parse_explanation(raw_output):
    for line in raw_output.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw_output


print("Functions loaded.")

Functions loaded.


## Load Data

We take 100 000 rows from the full dataset and keep only `text_block` and metadata columns. Short paragraphs (under 15 words) are dropped as they rarely contain enough context for reliable annotation.

In [4]:
RANDOM_SEED  = 42
N_LOAD       = 100_000
MIN_WORDS    = 15

df_full = pd.read_csv(
    "dataset/all_multi_paragraphs_2022_2026.csv",
    usecols=["article_id", "pub", "par_index", "text_block", "group"],
    nrows=N_LOAD
)

# drop rows where text_block is missing or too short
df_full = df_full.dropna(subset=["text_block"])
df_full = df_full[df_full["text_block"].str.split().str.len() >= MIN_WORDS]
df_full = df_full.reset_index(drop=True)

print(f"Rows after filtering: {len(df_full)}")
print(df_full["group"].value_counts().head(10))

Rows after filtering: 99891
group
REFUG    25136
UKR      10066
FRA       6895
RUS       6305
GBR       5443
USA       3831
HUN       3826
ITA       3575
NLD       3363
JUDE      3258
Name: count, dtype: int64


## Step 2.1 — Test Draft Annotation Instructions

We annotate a targeted sample of 50 rows from groups most likely to contain security frames. This is equivalent to the `bacchuss()` call in the R vignette.

After running: which cases were correctly labelled? Are there cases the instructions do not cover? Note problems below and update the instructions cell.

In [14]:
SAMPLE_SIZE_2_1 = 50
OUTPUT_2_1      = "results/annotation_step2_1.csv"

os.makedirs("results", exist_ok=True)

# groups most likely to carry security/crime frames
crime_relevant_groups = ["REFUG", "ASYL", "MIGR", "XKX", "SYR", "AFG"]
targeted = df_full[df_full["group"].isin(crime_relevant_groups)]

sample_2_1 = targeted.sample(n=SAMPLE_SIZE_2_1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Sample size: {len(sample_2_1)}")
print(sample_2_1["group"].value_counts())

Sample size: 50
group
REFUG    42
MIGR      4
ASYL      3
AFG       1
Name: count, dtype: int64


In [15]:
results_2_1 = []

for i, row in sample_2_1.iterrows():
    raw = annotate(str(row["text_block"]), instructions, reminder, temperature=0.0)
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "article_id":  row["article_id"],
        "par_index":   row["par_index"],
        "group":       row["group"],
        "text_block":  row["text_block"],
        "raw_output":  raw,
        "label":       label,
        "explanation": explanation
    })

    if (i + 1) % 10 == 0:
        sec_neg = sum(1 for r in results_2_1 if r["label"] == "SECURITY_NEGATIVE")
        sec_pos = sum(1 for r in results_2_1 if r["label"] == "SECURITY_POSITIVE")
        print(f"  [{i+1}/{SAMPLE_SIZE_2_1}] SECURITY_NEGATIVE: {sec_neg}  SECURITY_POSITIVE: {sec_pos}")

annotation_2_1 = pd.DataFrame(results_2_1)
annotation_2_1.to_csv(OUTPUT_2_1, index=False)

print("\nDone.")
print(annotation_2_1["label"].value_counts())

  [10/50] SECURITY_NEGATIVE: 0  SECURITY_POSITIVE: 0
  [20/50] SECURITY_NEGATIVE: 1  SECURITY_POSITIVE: 0
  [30/50] SECURITY_NEGATIVE: 2  SECURITY_POSITIVE: 0
  [40/50] SECURITY_NEGATIVE: 3  SECURITY_POSITIVE: 0
  [50/50] SECURITY_NEGATIVE: 3  SECURITY_POSITIVE: 0

Done.
label
NOT_SECURITY_CRIME    47
SECURITY_NEGATIVE      3
Name: count, dtype: int64


In [16]:
total = len(annotation_2_1)
print("=== Label Distribution ===")
for label, count in annotation_2_1["label"].value_counts().items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

unclear = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"]
if len(unclear) > 0:
    pct = len(unclear) / total * 100
    print(f"\nUNCLEAR labels: {len(unclear)} ({pct:.1f}%)")
    if pct > 5:
        print("   more than 5% unclear — adjust reminder and output format")
    print(unclear[["text_block", "raw_output"]].to_string())
else:
    print("\nNo unclear labels.")

=== Label Distribution ===
  NOT_SECURITY_CRIME: 47 (94.0%)
  SECURITY_NEGATIVE: 3 (6.0%)

No unclear labels.


In [17]:
# visual inspection — sorted so security frames appear first
annotation_2_1.sort_values("label")[["group", "label", "explanation", "text_block"]]

,group,label,explanation,text_block
0,REFUG,NOT_SECURITY_CRIME,Der Absatz thematisiert völkerrechtswidrige An...,Moskau Die russischen Besatzer haben die Schei...
26,REFUG,NOT_SECURITY_CRIME,"Der Absatz thematisiert zwar Grenzkontrollen, ...",Seit Ende Februar sind rund eine Million [Grup...
27,REFUG,NOT_SECURITY_CRIME,Der Absatz beschreibt eine **humanitäre Maßnah...,Umzug von Leipzig nach Wermsdorf\n\nSchon ...
28,MIGR,NOT_SECURITY_CRIME,Der Absatz thematisiert die rechtliche und adm...,"\n\nSie sind gut integriert, leben seit Jahren..."
29,REFUG,NOT_SECURITY_CRIME,Der Absatz behandelt primär Themen wie Arbeits...,Ziel der Arbeitsagentur ist es nach Angaben ih...
30,REFUG,NOT_SECURITY_CRIME,Der Absatz diskutiert **ausschließlich** die *...,Aus Sicht von Thies eignet sich der Standort h...
31,REFUG,NOT_SECURITY_CRIME,Der Absatz thematisiert zwar militärische Konf...,"Die Entscheidung der Nato, eine Flugverbotszon..."
32,REFUG,NOT_SECURITY_CRIME,Der Absatz behandelt ausschließlich administra...,\n\nAuf rund 2500 Personen hat sich die Zahl d...
33,REFUG,NOT_SECURITY_CRIME,Der Absatz thematisiert die humanitäre Unterst...,Mit einer Spendenaktion zeigte der Kreisverban...
34,ASYL,NOT_SECURITY_CRIME,Der Absatz erwähnt zwar Grenzkontrollen und ei...,Kontrollen mitten in Europa: An 27 Übergängen ...


### Notes from 2.1

Write down what you see before moving on. What went wrong, what was borderline, what is not covered by the instructions yet.

| Round | Problem | Decision Rule Added |
|-------|---------|--------------------|
| 1     |         |                    |
| 2     |         |                    |

## Step 2.2 — Identify Hard Cases

We run the same 20 rows 5 times at higher temperature and look for disagreement across runs. Cases with agreement below 80% are the ones where the instructions leave too much room for interpretation. These are the cases to focus on when revising.

This is equivalent to the `briseus()` call in the R vignette.

In [18]:
SAMPLE_SIZE_2_2 = 20
N_RUNS          = 5
TEMPERATURE_2_2 = 0.7
OUTPUT_2_2      = "results/annotation_step2_2.csv"

sample_2_2 = targeted.sample(n=SAMPLE_SIZE_2_2, random_state=RANDOM_SEED + 1).reset_index(drop=True)

print(f"Sample size: {len(sample_2_2)} rows, {N_RUNS} runs each")

Sample size: 20 rows, 5 runs each


In [19]:
from collections import Counter

results_2_2 = []

for i, row in sample_2_2.iterrows():
    run_labels = []
    for r in range(N_RUNS):
        raw = annotate(str(row["text_block"]), instructions, reminder, temperature=TEMPERATURE_2_2)
        run_labels.append(parse_label(raw))

    counts = Counter(run_labels)
    final_label = counts.most_common(1)[0][0]
    agreement = counts.most_common(1)[0][1] / N_RUNS

    entry = {
        "article_id":  row["article_id"],
        "par_index":   row["par_index"],
        "group":       row["group"],
        "text_block":  row["text_block"],
        "final_label": final_label,
        "agreement":   agreement,
    }
    for r in range(N_RUNS):
        entry[f"run_{r+1}"] = run_labels[r]

    results_2_2.append(entry)
    print(f"  row {i+1}/{SAMPLE_SIZE_2_2} — {final_label} (agreement {agreement:.0%})")

annotation_2_2 = pd.DataFrame(results_2_2)
annotation_2_2 = annotation_2_2.sort_values("agreement").reset_index(drop=True)
annotation_2_2.to_csv(OUTPUT_2_2, index=False)

print("\nDone.")

  row 1/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 2/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 3/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 4/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 5/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 6/20 — SECURITY_NEGATIVE (agreement 60%)
  row 7/20 — SECURITY_NEGATIVE (agreement 100%)
  row 8/20 — NOT_SECURITY_CRIME (agreement 80%)
  row 9/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 10/20 — SECURITY_POSITIVE (agreement 100%)
  row 11/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 12/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 13/20 — NOT_SECURITY_CRIME (agreement 80%)
  row 14/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 15/20 — NOT_SECURITY_CRIME (agreement 80%)
  row 16/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 17/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 18/20 — NOT_SECURITY_CRIME (agreement 100%)
  row 19/20 — NOT_SECURITY_CRIME (agreement 80%)
  row 20/20 — NOT_SECURITY_CRIME (agreement 100%)

Done.


In [20]:
run_cols = [f"run_{r+1}" for r in range(N_RUNS)]

print("=== Full Results (sorted by agreement, lowest first) ===")
annotation_2_2[["agreement", "final_label", "text_block"] + run_cols]

=== Full Results (sorted by agreement, lowest first) ===


,agreement,final_label,text_block,run_1,run_2,run_3,run_4,run_5
0,0.6,SECURITY_NEGATIVE,"\n\nDie Villen sind vom Feinsten, die Wohnunge...",SECURITY_NEGATIVE,NOT_SECURITY_CRIME,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,SECURITY_NEGATIVE
1,0.8,NOT_SECURITY_CRIME,"Pit Clausen, Vorsitzender des Städtetages NRW,...",NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,SECURITY_NEGATIVE,NOT_SECURITY_CRIME
2,0.8,NOT_SECURITY_CRIME,\n\nFünf Schüsse aus der Maschinenpistole eine...,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,SECURITY_NEGATIVE
3,0.8,NOT_SECURITY_CRIME,Das Regime drängt [Gruppe 1] aus dem öffentlic...,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME
4,0.8,NOT_SECURITY_CRIME,""" NRW schickt [Gruppe 1] ohne Tuberkulose-Test...",NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,SECURITY_NEGATIVE,NOT_SECURITY_CRIME
5,1.0,NOT_SECURITY_CRIME,Sie kamen aus den Landesunterkünften in Booste...,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME
6,1.0,NOT_SECURITY_CRIME,Erfurt In Thüringen sind derzeit mehrere Landk...,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME
7,1.0,NOT_SECURITY_CRIME,"Das heißt?\n\nDas heißt, dass auch ukrainische...",NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME
8,1.0,SECURITY_NEGATIVE,In Mexiko fordern gewalttätige Drogenkartelle ...,SECURITY_NEGATIVE,SECURITY_NEGATIVE,SECURITY_NEGATIVE,SECURITY_NEGATIVE,SECURITY_NEGATIVE
9,1.0,NOT_SECURITY_CRIME,Berlin Es ist die größte Fluchtbewegung in Eur...,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME,NOT_SECURITY_CRIME


In [21]:
hard_cases = annotation_2_2[annotation_2_2["agreement"] < 0.8]
print(f"Hard cases (agreement < 80%): {len(hard_cases)} out of {len(annotation_2_2)}")

for _, row in hard_cases.iterrows():
    runs = [row[f"run_{r+1}"] for r in range(N_RUNS)]
    print(f"\n{'='*60}")
    print(f"Agreement: {row['agreement']:.0%}  |  Final label: {row['final_label']}")
    print(f"Runs: {runs}")
    print(f"Group: {row['group']}")
    print(f"Text: {str(row['text_block'])[:500]}...")
    print("TODO: Why is this hard? What decision rule would resolve it?")

Hard cases (agreement < 80%): 1 out of 20

Agreement: 60%  |  Final label: SECURITY_NEGATIVE
Runs: ['SECURITY_NEGATIVE', 'NOT_SECURITY_CRIME', 'SECURITY_NEGATIVE', 'NOT_SECURITY_CRIME', 'SECURITY_NEGATIVE']
Group: REFUG
Text: 

Die Villen sind vom Feinsten, die Wohnungen luxuriös. In London gibt es außergewöhnlich viele Immobilien russischer Oligarchen.  Bürgermeister Sadiq Khan hat sich nun dafür ausgesprochen, ukrainische [Gruppe 1] in diesen Häusern und Wohnungen unterzubringen. Ein Großteil davon stünden ohnehin leer, sagte Khan dem Times Radio am Montag. Der Labour-Politiker geht davon aus, dass viele Immobilien russischer Superreicher eher zur Geldwäsche gekauft wurden, als um darin zu wohnen. Es handle sich um...
TODO: Why is this hard? What decision rule would resolve it?


In [22]:
print("=== Agreement Distribution ===")
print(annotation_2_2["agreement"].describe())
print(f"\nFull agreement (100%): {len(annotation_2_2[annotation_2_2['agreement'] == 1.0])} rows")
print(f"Hard cases (<80%):     {len(annotation_2_2[annotation_2_2['agreement'] < 0.8])} rows")
print(f"Very hard (<60%):      {len(annotation_2_2[annotation_2_2['agreement'] < 0.6])} rows")

=== Agreement Distribution ===
count    20.000000
mean      0.940000
std       0.114248
min       0.600000
25%       0.950000
50%       1.000000
75%       1.000000
max       1.000000
Name: agreement, dtype: float64

Full agreement (100%): 15 rows
Hard cases (<80%):     1 rows
Very hard (<60%):      0 rows


## Next Steps

1. Note problems in the table below
2. Update the instructions cell and add/clarify decision rules
3. Rerun 2.1 and 2.2 with the updated instructions
4. Stop when your instructions broadly cover the cases you see, even if reliability is not perfect yet

Once coverage is broad, move to Step 3: testing on a curated sample with human gold-standard labels, chain-of-thought prompting, and few-shot examples.

### Notes (fill in as you iterate)

| Round | Problem Identified | Decision Rule Added |
|-------|--------------------|---------------------|
| 1     |                    |                     |
| 2     |                    |                     |
| 3     |                    |                     |